# arc3-arena-p50h-fork (S19 build, A1h = A1 + prefetch)
Exact fork of the public **artificialagencylab.com** runtime `richardcsaky/arc-arena-hero-v2-blocks2-p50-int3-runtime` (uploaded 30 Sep 2026;
team rank 7, best public LB 38.62 on 8 Oct, 37.94 by 5 Oct). Their code is not modified. This notebook only
selects the bundle, checks its sha256 and prints mount diagnostics.

What it runs (from the bundle's `configs/comparison.json`):
- Qwen3.8-Flash-Next NVFP4 base (RadixArk via keithtyser vLLM runtime), routed experts replaced by an ARC-calibrated
  int3 g16 overlay (`richardcsaky/arc-expert-int3-g16-v1`, manifest sha256 7fca7969...), 50% of routed experts pruned per layer
- int4 KV cache, MTP off, 76 concurrent game streams, 98,304-token window, completion cap 16,384, temperature 0.7
- 2-block schedule: 30,500 s play wall as 2 blocks of 15,250 s, roster split 76 / 34; a level-0 game retires at 90K
  completion tokens, each cleared level adds 90K; freed slots go to never-run games, then to parked games with the most levels
  (42 spare refill slots in block 2)
- Duck-transfer harness with harness_fixes (exact compaction trigger, summary guard, python inventory, cap follow-up,
  stuck notice, sandbox, timeout interrupt, prompt text, summary headroom, inventory imports) and `run_plan` multi-step actions

Save run (not a rerun): setup plus a 300 s public preview on 110 cyclic clones. Envelope 5,400 s instead of their
4,200 s: the preview needs setup <= envelope - 600 s cleanup - 300 s, and their setup guard allows up to 3,600 s, so
4,200 s could refuse the preview on a slow mount. The rerun ignores this value (bootstrap_arena.py sets the envelope to
submission_seconds when KAGGLE_IS_COMPETITION_RERUN is set), so the scored run is unchanged. Expected cost ~0.5 GPU-h
(two public runb save runs measured setup at 1,164 and 1,245 s on this image and GPU).
Competition rerun: 32,400 s envelope from notebook start, 600 s cleanup reserve, gateway roster.
Pinned image is the one their public notebook used (Python 3.12).

S19 change (A1h): one daemon thread pool prefetches the checkpoint files into page cache in vLLM's load order, at most 10 GB ahead of the loader (position parsed from the vLLM log). A1 loaded 135 GB as a single NFS stream (1,031 s); A2 failed its 3,000 s ready guard on a slower host. Engine, bundle and sha256 are unchanged.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, tarfile, time
assert sys.version_info[:2] == (3, 12), 'Select the Python 3.12 notebook image'
NOTEBOOK_START_EPOCH = time.time()
try:  # S18 diagnostics only (no behaviour change; can never fail the run): GPU, mounts, rerun flag
    print(json.dumps({'python': sys.version.split()[0], 'rerun': os.environ.get('KAGGLE_IS_COMPETITION_RERUN', '')}))
    for root in (Path('/kaggle/input'), Path('/kaggle/input/datasets'), Path('/kaggle/input/models')):
        if root.is_dir():
            print(root, sorted(p.name for p in root.iterdir())[:40])
    subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], check=False, timeout=60)
except Exception as exc:
    print('diagnostics skipped:', repr(exc)[:200])
roots = [Path('/kaggle/input/datasets/richardcsaky/arc-arena-hero-v2-blocks2-p50-int3-runtime'), Path('/kaggle/input/arc-arena-hero-v2-blocks2-p50-int3-runtime')]
mount = next((p for p in roots if p.is_dir()), None)
assert mount is not None, 'Attach the prepared richardcsaky/arc-arena-hero-v2-blocks2-p50-int3-runtime dataset'
blob = mount / 'arena-flash-competition-blocks2-p50-int3-v1.tar.gz.blob'
with blob.open('rb') as stream:
    assert hashlib.file_digest(stream, 'sha256').hexdigest() == 'f76187cea700749ab7572b952a9ae5e0038dfbec83619bf706e720cc9c9db286', 'Runtime archive differs from reviewed build'
bundle = Path('/tmp/arena-runtime')
bundle.mkdir(exist_ok=False)
with tarfile.open(blob, 'r:gz') as archive:
    archive.extractall(bundle, filter='data')
def _arc_prefetch(threads=8, window=10e9, max_seconds=3300):
    # S19 A1h: bounded parallel page-cache prefetch of the vLLM checkpoint. Never raises.
    import json, os, re, threading, time
    from pathlib import Path
    t0 = time.time(); stats = {'threads': threads, 'window_bytes': window, 'files': 0, 'bytes': 0, 'done_files': 0,
                               'done_bytes': 0, 'errors': 0, 'events': []}
    out = Path('/kaggle/working/arena-prefetch.json')
    def note(ev):
        stats['events'].append([round(time.time() - t0, 1), ev]); print('ARC_PREFETCH', json.dumps(ev), flush=True)
    try:
        base = next((p for p in (Path('/kaggle/input/models/keithtyser/qwen3-8-flash-next-nvfp4/pytorch/radixark-modelopt-fp4/1'),
                                 Path('/kaggle/input/qwen3-8-flash-next-nvfp4/pytorch/radixark-modelopt-fp4/1')) if p.is_dir()), None)
        over = next((p for p in (Path('/kaggle/input/datasets/richardcsaky/arc-expert-int3-g16-v1'),
                                 Path('/kaggle/input/arc-expert-int3-g16-v1')) if p.is_dir()), None)
        if base is None:
            note({'skip': 'base model mount not found'}); return
        names = sorted(set(json.loads((base / 'model.safetensors.index.json').read_text())['weight_map'].values()),
                       key=lambda s: [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', s)])
        files = [(over / n) if (over is not None and (over / n).is_file()) else (base / n) for n in names]
        sizes = [f.stat().st_size for f in files]
        cum = [0]
        for s in sizes: cum.append(cum[-1] + s)
        stats.update(files=len(files), bytes=cum[-1], overlay=str(over), base=str(base))
        note({'start': len(files), 'gb': round(cum[-1] / 1e9, 1)})
        logs = [Path('/tmp/arena/serving/vllm-openai-server.log'), Path('/kaggle/working/arena/serving/vllm-openai-server.log')]
        prog = re.compile(rb'(\d+)/(\d+) \[')
        state = {'pos': 0, 'next': 0}
        lock = threading.Lock()
        def loader_pos():
            for lp in logs:
                try:
                    with open(lp, 'rb') as fh:
                        fh.seek(max(0, lp.stat().st_size - 8192)); tail = fh.read()
                    m = [x for x in prog.findall(tail) if int(x[1]) == len(files)]
                    if m: return int(m[-1][0])
                except Exception:
                    pass
            return 0
        def worker():
            buf = bytearray(16 << 20); mv = memoryview(buf)
            while time.time() - t0 < max_seconds:
                with lock:
                    i = state['next']
                    if i >= len(files): return
                    if cum[i] - cum[state['pos']] > window:
                        i = None
                    else:
                        state['next'] += 1
                if i is None:
                    time.sleep(1); continue
                try:
                    with open(files[i], 'rb', buffering=0) as fh:
                        while fh.readinto(mv): pass
                    with lock:
                        stats['done_files'] += 1; stats['done_bytes'] += sizes[i]
                except Exception as exc:
                    stats['errors'] += 1
                    if stats['errors'] < 5: note({'error': repr(exc)[:160], 'file': files[i].name})
        def monitor():
            last = -1
            while time.time() - t0 < max_seconds and stats['done_files'] < len(files):
                state['pos'] = min(loader_pos(), len(files))
                el = time.time() - t0
                if int(el) // 60 != last:
                    last = int(el) // 60
                    note({'t': round(el), 'loader': state['pos'], 'prefetched': stats['done_files'],
                          'gb': round(stats['done_bytes'] / 1e9, 1), 'mb_s': round(stats['done_bytes'] / 1e6 / max(el, 1), 1)})
                    try: out.write_text(json.dumps(stats))
                    except Exception: pass
                time.sleep(2)
        pool = [threading.Thread(target=worker, daemon=True) for _ in range(threads)]
        for th in pool: th.start()
        mon = threading.Thread(target=monitor, daemon=True); mon.start()
        for th in pool: th.join()
        el = time.time() - t0
        note({'finished': stats['done_files'], 'gb': round(stats['done_bytes'] / 1e9, 1), 'seconds': round(el),
              'mb_s': round(stats['done_bytes'] / 1e6 / max(el, 1), 1)})
    except Exception as exc:
        note({'fatal': repr(exc)[:200]})
    try: out.write_text(json.dumps(stats))
    except Exception: pass

try:
    import threading as _th
    _th.Thread(target=_arc_prefetch, daemon=True).start()
except Exception as _exc:
    print('prefetch not started:', repr(_exc)[:200])
command = [sys.executable, str(bundle / 'bootstrap_arena.py'), '--bundle', str(bundle), '--working', '/tmp/arena',
           '--output', '/kaggle/working/arena', '--max-seconds', '5400', '--notebook-start-epoch', str(NOTEBOOK_START_EPOCH)]
result = subprocess.run(command, check=False)
print(json.dumps({'bootstrap_exit': result.returncode, 'elapsed': time.time() - NOTEBOOK_START_EPOCH}))


In [ ]:
import json, os
from pathlib import Path
EXPECTED_PREVIEW_TRIALS = 1
rerun = os.environ.get('KAGGLE_IS_COMPETITION_RERUN', '').lower() in ('1', 'true')
report_path = Path('/kaggle/working/arena/trials/study.json')
if not report_path.is_file():
    raise RuntimeError(f'Arena stopped before final report: bootstrap_exit={result.returncode}')
record = json.loads(report_path.read_text())
acceptable = {'complete', 'complete_with_preemptions', 'error'} if rerun else {'complete'}
trials = record.get('trials', [])
expected = 1 if rerun else EXPECTED_PREVIEW_TRIALS
# On the hidden rerun the gateway scorecard is already closed: only a missing or unacceptable
# gateway trial fails the notebook, never the bootstrap exit code or the preview-only checks.
if len(trials) != expected or any(t.get('status') not in acceptable for t in trials) or (result.returncode and not rerun):
    raise RuntimeError('Arena validation failed; inspect saved runtime and trial reports')
for row in record.get('comparison', []):
    print(json.dumps(row))
if not rerun:
    import pandas as pd
    pd.DataFrame([['1_0', '1', True, 1]],
                 columns=['row_id', 'game_id', 'end_of_game', 'score']).to_parquet(
                     '/kaggle/working/submission.parquet', index=False)
    print('Public validation completed. The placeholder selects the notebook for a hidden rerun.')
else:
    print('Hidden gateway scorecard closed; the competition gateway produces the submission artifact.')
